# 14 — G-Eval (LLM-as-a-Judge, backfill sulla split test)

Aggiunge al benchmark una metrica **LLM-as-a-Judge**: G-Eval (Liu et al., 2023) fa valutare a
un modello linguistico ogni riassunto generato su quattro dimensioni, in scala **1-5**:

| dimensione | cosa misura |
|---|---|
| **coherence** | il riassunto e' ben strutturato e si legge come un testo unico |
| **consistency** | fedelta' fattuale rispetto alla sorgente (allucinazioni, contraddizioni) |
| **fluency** | correttezza grammaticale e scorrevolezza |
| **relevance** | quanta informazione importante della sorgente viene catturata |

E' l'unica metrica del benchmark che non passa dal riferimento umano: ROUGE/BLEU/METEOR
misurano **sovrapposizione lessicale** con il `summary` di riferimento e BERTScore
**similarita' semantica** con lo stesso. Nessuna delle due dice se un riassunto e' coerente o
se ha inventato dei fatti — cioe' proprio le dimensioni su cui i sistemi astrattivi
differiscono davvero.

## Perche' non `psr.g_eval()`

`pyAutoSummarizer` (gia' usato per ROUGE/BLEU/METEOR, vedi `summ_utils.crea_valutatore`)
espone gia' un G-Eval. Ispezionandone il codice (`pyAutoSummarizer/base/psr.py`, v1.2.0)
risulta inutilizzabile qui per **quattro** motivi:

1. costruisce `openai.OpenAI(api_key=...)` **senza `base_url`**: parla solo con OpenAI, non
   con Azure (ne' con l'endpoint ollama locale usato dai notebook 07-09);
2. ha `max_tokens=5` e `temperature=0.0` **cablati**, entrambi fatali per un modello
   reasoning (che rifiuta `temperature` e spende il budget in token di ragionamento);
3. legge la sorgente da `self.full_txt`, cioe' dal testo passato al costruttore: servirebbe
   una `psr.summarization(sorgente)` nuova — e pesante — per **ogni** esempio, e il pattern a
   istanza fittizia condivisa di `crea_valutatore()` non regge;
4. fa **una chiamata API per dimensione**, quadruplicando il costo.

Qui la funzione viene quindi reimplementata in `summ_utils.py`, ma le **rubriche restano
verbatim**: `su.RUBRICHE_GEVAL` e' derivato meccanicamente da `su.PROMPT_GEVAL_ORIGINALI`
(copia letterale dei template della libreria) tagliando la coda `"Reply with a single digit
only."`, qui sostituita dalla richiesta di un unico oggetto JSON. La metrica resta cosi'
difendibile come *il G-Eval di pyAutoSummarizer, reimplementato per un endpoint non-OpenAI*.

## Scelta del giudice: `gpt-5.4-mini` su Azure AI Foundry

Deployment **GlobalStandard** dedicato su `agirasella-resource` (swedencentral). Due proprieta'
lo motivano:

1. **Indipendente da tutti e 18 i metodi valutati**: nessun self-judging. Usare `gpt-5-mini`
   (il modello del notebook 12, presente nel benchmark come metodo `gpt5mini`) significherebbe
   farlo giudicare i propri output, con il noto bias di auto-preferenza.
2. **Piu' recente di ogni generatore del benchmark** (`gpt-5-mini` e' 2025-08-07,
   `gpt-5.4-mini` e' 2026-03-17). Conta soprattutto per **consistency**: individuare
   un'allucinazione sottile dentro un cluster multi-documento e' un compito di ragionamento, e
   la correlazione giudice-umano scala con la capacita' del giudice proprio su quella
   dimensione. Un giudice piu' debole tenderebbe a schiacciare i punteggi verso l'alto,
   penalizzando poco i modelli forti.

Verificato dal vivo sulla sottoscrizione: DeepSeek/Grok/Llama/Mistral **non sono deployabili**
su questo account AIServices (stessa ragione per cui i notebook Claude Haiku e DeepSeek furono
abbandonati), e **`gpt-5.1-mini` non esiste** — la linea 5.1 e' `gpt-5.1` / `-chat` /
`-codex*`.

### Conseguenza: e' un modello reasoning

Valgono le stesse regole del notebook 12: **niente `temperature`** (accetta solo il default),
`max_completion_tokens` al posto di `max_tokens`, `reasoning_effort='minimal'`. I token di
ragionamento consumano il budget **prima** dell'output visibile: e' lo stesso modo di fallire
gia' visto con gemma (notebook 08) e gpt-5-mini (notebook 12), per questo il budget e' 1500 e
non 200. Le corse **non sono riproducibili bit-a-bit**: l'artefatto di riproducibilita' e' la
cache JSONL committata, da cui le metriche si riderivano a costo zero.

## Protocollo

- **una chiamata per (metodo, riga)**, con tutte e quattro le dimensioni in un unico JSON
  (`response_format` con `json_schema` strict: il formato e' garantito lato server);
- sorgente troncata a **3.500 parole** (`su.MAX_PAROLE_SORGENTE_GEVAL`): lascia **intero il
  91,5%** dei cluster della split test (mediana 1.288 parole, p90 3.244), con un tetto solo sui
  casi estremi (fino a 35.362 parole). E' molto piu' di quanto vedano BART/PEGASUS (1.024 token);
- ambito: **split test intera**, 18 metodi su 5.610 righe con copertura non uniforme =
  **~100.600 giudizi** (72.681 dei 13 metodi originali, gia' in cache, piu' i ~27.940 dei
  cinque metodi dei notebook 15-17 aggiunti col backfill dell'issue #12);
- tier standard concorrente (niente Batch API: `gpt-5.4-mini` la supporta, ma il ritardo di
  24 h per job e i limiti di 200 MB/file la rendono piu' scomoda della prompt cache).

## Ambito e output

**Backfill una tantum**, come il notebook 13: non tocca i notebook 01-04/06-12 ne' il loro
ciclo di generazione. I 18 metodi sono quelli della Vista 2 del notebook 05.

I punteggi vanno in **file separati** (`{metodo}_test_geval_per_example.csv` e
`..._geval_aggregate.json`), **mai** uniti al CSV per-esempio standard. Il motivo e' in
`summ_utils.valuta_e_salva`, la cui media interna somma **ogni** colonna su **ogni** riga: il
giudice lascia scoperte alcune righe (content filter di Azure, parsing fallito) e la media
esploderebbe con un `KeyError`; e restringere le righe valutate riscriverebbe i CSV gia'
committati, cambiando medie e `n_esempi` di ROUGE/BLEU/METEOR/BERTScore.

In [1]:
# --- Dipendenze --------------------------------------------------------------
# Su Colab la libreria openai potrebbe non esserci: installazione idempotente.
try:
    import openai  # noqa: F401
except ImportError:
    %pip install -q openai
    import openai  # noqa: F401

In [2]:
# --- Configurazione ----------------------------------------------------------
import json
import os
import random
import time
from collections import Counter

import summ_utils as su

SCOPE = os.environ.get('GEVAL_SCOPE', 'test')

GIUDICE = 'gpt-5.4-mini'
# Su Azure OpenAI `model=` e' il nome del DEPLOYMENT, non del modello
DEPLOYMENT_GIUDICE = os.environ.get('AZURE_GEVAL_DEPLOYMENT', 'gpt-5.4-mini')
AZURE_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']   # radice della risorsa, senza path
AZURE_API_KEY = os.environ['AZURE_OPENAI_API_KEY']
# API "v1" di Azure OpenAI: nessuna api-version datata (come nel notebook 12)
BASE_URL = AZURE_ENDPOINT.rstrip('/') + '/openai/v1/'

# Parametri del giudice. gpt-5.4-mini e' un modello REASONING:
#  - `temperature` non e' accettata (solo il default)
#  - i token di reasoning consumano il budget PRIMA dell'output visibile, quindi
#    1500 e non 200 (stesso valore gia' validato nel notebook 12)
MAX_COMPLETION_TOKENS = 1500
REASONING_EFFORT = 'minimal'
MAX_PAROLE = su.MAX_PAROLE_SORGENTE_GEVAL   # 3500 -> 91,5% dei cluster interi

# Concorrenza e controllo di spesa (sovrascrivibili da scripts/run_geval.py)
# ATTENZIONE: pochi thread costano MENO. La prompt cache di Azure e' per istanza di
# backend e un deployment GlobalStandard distribuisce le richieste: con piu' righe in
# volo le chiamate successive alla prima di una riga atterrano su istanze fredde.
# Misurato sul pilota: 76% di hit a riga singola (~$56 sulla corsa) contro 33% a
# 8 thread (~$111).
N_THREAD = int(os.environ.get('GEVAL_THREAD', 2))
LIMIT_RIGHE = int(os.environ['GEVAL_RIGHE']) if os.environ.get('GEVAL_RIGHE') else None
BUDGET_MASSIMO = float(os.environ['GEVAL_BUDGET']) if os.environ.get('GEVAL_BUDGET') else None
RIPROVA_ERRORI = os.environ.get('GEVAL_RIPROVA_ERRORI') == '1'
SOLO_METRICHE = os.environ.get('GEVAL_SOLO_METRICHE') == '1'
PILOTA = int(os.environ['GEVAL_PILOTA']) if os.environ.get('GEVAL_PILOTA') else 0
OGNI = int(os.environ.get('GEVAL_OGNI', 1500))   # cadenza del blocco costo
# Ordine delle righe. 'casuale' (seed fisso) e' consigliato quando si imposta un
# budget che fermera' la corsa a meta': cosi' quello che si riesce a giudicare e'
# un campione casuale della split test invece del suo primo tratto, e le medie
# restano stime non distorte. Con la cache l'ordine non influisce sulla ripresa.
ORDINE = os.environ.get('GEVAL_ORDINE', 'corpus')   # 'corpus' | 'casuale'
SEED_ORDINE = 42

BASE = su.trova_base_dir()
P = su.percorsi_standard(BASE)

# Stessi 18 metodi con metriche 'test' del notebook 05 (Vista 2) e del notebook 13.
# I cinque dei notebook 15-17 sono stati aggiunti col backfill dell'issue #12: la
# cache e' per (metodo, row_id), quindi allargare la lista fa giudicare SOLO i nuovi.
METODI_BASELINE = ['firstk_psr', 'firstk_nltk',          # notebook 10 (First-k)
                   'centroid_mmr', 'centroid_mmr_bert']  # notebook 11 (Centroid+MMR)
METODI_NON_SUP = ['lsa', 'lsa_steinberger',        # notebook 15 (LSA/SVD, due selezioni)
                  'sbert_kmeans', 'sbert_agglom',  # notebook 16 (clustering su embedding SBERT)
                  'lda']                           # notebook 17 (topic modeling LDA)
METODI = METODI_BASELINE + ['textrank', 'lexrank', 'bart', 'pegasus', 'primera',
                            'qwen', 'gemma', 'mistral'] + ['gpt5mini'] + METODI_NON_SUP


def percorso_riassunti(metodo):
    """TextRank/LexRank non hanno una corsa '_test.tsv' dedicata: i riassunti sono
    nel file '_full.tsv' (intero complete.tab). I riferimenti sotto sono comunque gia'
    ristretti alla sola split test, quindi il filtro avviene automaticamente."""
    suffisso = 'full' if metodo in ('textrank', 'lexrank') else 'test'
    return P['summaries_dir'] / f'{metodo}_{suffisso}.tsv'


CACHE_PATH = P['metrics_dir'] / f'geval_cache_{SCOPE}.jsonl'

# Prezzi unitari dal listino Azure (fallback: su.PREZZI_GEVAL). Non esiste un'API
# Azure di costo in tempo reale: il costo si calcola dai token dell'oggetto `usage`.
# La valuta va fatta corrispondere a quella di fatturazione della sottoscrizione
# (default USD, ma il listino EUR di Azure NON e' una conversione al cambio -
# e' un listino a se stante, ~0,8776x il numero USD su ogni meter, verificato
# confrontando il consumo di credito reale con la stima in $ della prima corsa).
VALUTA = os.environ.get('GEVAL_VALUTA', 'USD')
PREZZI = su.prezzi_retail_azure(valuta=VALUTA)

config = {
    'giudice': GIUDICE,
    'deployment': DEPLOYMENT_GIUDICE,
    'backend': 'Azure AI Foundry / Azure OpenAI (chat completions, rotta v1)',
    'max_completion_tokens': MAX_COMPLETION_TOKENS,
    'reasoning_effort': REASONING_EFFORT,
    'temperature': 'non impostata (modello reasoning: accetta solo il default)',
    'max_parole_sorgente': MAX_PAROLE,
    'protocollo': 'una chiamata per (metodo, riga), 4 dimensioni in un unico JSON',
    'response_format': 'json_schema strict',
    'rubriche': 'verbatim da pyAutoSummarizer 1.2.0 (psr.summarization.g_eval)',
    'n_thread': N_THREAD,
    'valuta': VALUTA, 'prezzi_per_1M_token': PREZZI,
    'nota': ('punteggi 1-5 su coherence/consistency/fluency/relevance; il giudice e\' '
             'indipendente da tutti i metodi valutati e piu\' recente di ogni generatore '
             'del benchmark'),
}

print(f'Scope        : {SCOPE}')
print(f'Giudice      : {GIUDICE} (deployment {DEPLOYMENT_GIUDICE})')
print(f'Cache        : {CACHE_PATH}')
print(f'Thread       : {N_THREAD}   report ogni {OGNI} giudizi')
print(f'Budget max   : {("$%.2f" % BUDGET_MASSIMO) if BUDGET_MASSIMO else "nessuno"}')
print(f'Limite righe : {LIMIT_RIGHE or "tutte"}   pilota: {PILOTA or "no"}')

Prezzi da Azure Retail Prices API (EUR/1M token): {'input': 0.659, 'cached': 0.0659, 'output': 3.9538}
Scope        : test
Giudice      : gpt-5.4-mini (deployment gpt-5.4-mini)
Cache        : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\geval_cache_test.jsonl
Thread       : 2   report ogni 1500 giudizi
Budget max   : $100.00
Limite righe : tutte   pilota: no


## Prompt e formato della risposta

Due vincoli guidano la costruzione dei messaggi, entrambi implementati in
`su.costruisci_messaggi_geval`:

**1. L'ordine e' un contratto, non uno stile.** Il messaggio `system` (le quattro rubriche +
la richiesta del JSON) e' **identico su tutte le chiamate**, e il messaggio `user`
comincia con la **sorgente troncata** — la stessa per tutti i metodi di una data riga —
mettendo in fondo il **riassunto**, unica parte che cambia:

```
system : <rubriche + formato JSON>          <- costante
user   : Source:\n<sorgente troncata>       <- costante dentro la riga
         \n\nSummary:\n<riassunto>          <- varia per metodo
```

Cosi' il **primo** giudizio di una riga popola la prompt cache di Azure e tutti quelli
successivi la riusano (~92% del prefisso in cache, pagato a tariffa ridotta). Invertire l'ordine —
riassunto prima della sorgente — azzererebbe il risparmio: e' la differenza fra ~$39 e ~$180
di soli token di input sulla corsa completa.

**2. Il formato lo garantisce il server.** `response_format` con `json_schema` in modalita'
`strict` (`su.SCHEMA_GEVAL`) obbliga il modello a produrre esattamente quattro interi. Nota:
in `strict` non sono ammessi `minimum`/`maximum`, quindi la scala 1-5 e' espressa con `enum`.
`su.estrai_punteggi_geval` resta comunque difensivo (code fence, testo attorno, chiavi
mancanti, valori fuori scala) e **solleva** invece di inventare un punteggio.

In [3]:
# --- Client Azure e singolo giudizio -----------------------------------------
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=AZURE_API_KEY)

print(su.PROMPT_GEVAL_SISTEMA)
print('\n' + '-' * 70)
print('Schema JSON:', json.dumps(su.SCHEMA_GEVAL))


def giudica_uno(sorgente, riassunto):
    """Un giudizio G-Eval -> (punteggi, usage). Solleva su risposta vuota o non conforme."""
    def chiamata():
        return client.chat.completions.create(
            model=DEPLOYMENT_GIUDICE,
            messages=su.costruisci_messaggi_geval(sorgente, riassunto, MAX_PAROLE),
            max_completion_tokens=MAX_COMPLETION_TOKENS,
            reasoning_effort=REASONING_EFFORT,
            response_format={'type': 'json_schema',
                             'json_schema': {'name': 'geval', 'strict': True,
                                             'schema': su.SCHEMA_GEVAL}})

    risposta = su.chiama_con_backoff(chiamata)
    scelta = risposta.choices[0]
    contenuto = scelta.message.content
    if not contenuto or not contenuto.strip():
        # budget di reasoning esaurito prima dell'output visibile: stesso modo di
        # fallire di gemma (nb 08) e gpt-5-mini (nb 12). Si solleva -> nessun
        # punteggio inventato, la riga resta ritentabile.
        raise RuntimeError(f'risposta vuota (finish_reason={scelta.finish_reason})')
    return su.estrai_punteggi_geval(contenuto), risposta.usage

You are an expert evaluator of automatically generated news summaries.
Rate the summary provided by the user on each of the following four dimensions, on a scale of 1 to 5.

COHERENCE
Rate the COHERENCE of the summary below on a scale of 1 to 5, where 1 = completely incoherent and 5 = perfectly coherent and well-structured.

CONSISTENCY
Rate the FACTUAL CONSISTENCY of the summary below with respect to the source on a scale of 1 to 5, where 1 = many hallucinations or contradictions and 5 = fully faithful to the source.

FLUENCY
Rate the FLUENCY and grammaticality of the summary below on a scale of 1 to 5, where 1 = unreadable and 5 = fluent and error-free.

RELEVANCE
Rate the RELEVANCE of the summary below to the source text on a scale of 1 to 5, where 1 = captures nothing important and 5 = captures all key information.

Return a single JSON object with exactly the four integer keys "coherence", "consistency", "fluency", "relevance". No other text.

-------------------------------------

## Preparazione dei compiti

Riferimenti e riassunti si caricano **una sola volta** (~5.610 righe + 18 dizionari), poi si
costruisce la lista dei compiti **ordinata per `row_id`, con i metodi raggruppati**.

Questo raggruppamento non e' cosmetico: `su.giudica_geval_concorrente` prende la **riga** come
unita' di lavoro ed esegue i giudizi di quella riga **in sequenza dentro lo stesso thread**, cosi' che
il primo scaldi la prompt cache e gli altri la colpiscano. Il parallelismo e' **tra** righe
diverse. Parallelizzare per singolo giudizio farebbe partire insieme tutte le chiamate della
stessa riga, mancando la cache tutte quante.

In [4]:
# --- Costruzione dei compiti -------------------------------------------------
riferimenti_test = list(su.itera_split(P['complete_tab'], SCOPE))
print(f'Riferimenti split {SCOPE}: {len(riferimenti_test)} righe')

riassunti = {}
for metodo in METODI:
    path = percorso_riassunti(metodo)
    riassunti[metodo] = su.carica_riassunti(path)
    if not riassunti[metodo]:
        print(f'  ATTENZIONE {metodo}: {path.name} non trovato o vuoto')

righe = []
for rif in riferimenti_test:
    rid = rif['row_id']
    coppie = [(m, riassunti[m][rid]) for m in METODI if rid in riassunti[m]]
    if coppie:
        righe.append((rid, su.prepara_documento(rif['document']), coppie))

if ORDINE == 'casuale':
    random.Random(SEED_ORDINE).shuffle(righe)
    print(f"Ordine righe: CASUALE (seed {SEED_ORDINE}) — una eventuale interruzione "
          f"per budget lascia un campione casuale, non un prefisso del corpus")

totale_giudizi = sum(len(c) for _, _, c in righe)
print(f'\nRighe con almeno un riassunto: {len(righe)}')
print(f'Giudizi totali attesi       : {totale_giudizi:,}')
print('\nCopertura per metodo:')
for metodo in METODI:
    print(f'  {metodo:20s} {len(riassunti[metodo]):5d}')

lunghezze = [len(s.split()) for _, s, _ in righe]
tagliate = sum(1 for n in lunghezze if n > MAX_PAROLE)
print(f'\nLunghezza sorgente (parole): mediana {sorted(lunghezze)[len(lunghezze)//2]}, '
      f'max {max(lunghezze):,}')
print(f'Righe troncate a {MAX_PAROLE} parole: {tagliate} / {len(righe)} '
      f'({tagliate/len(righe):.1%})')

Riferimenti split test: 5610 righe


Ordine righe: CASUALE (seed 42) — una eventuale interruzione per budget lascia un campione casuale, non un prefisso del corpus

Righe con almeno un riassunto: 5610
Giudizi totali attesi       : 100,621

Copertura per metodo:
  firstk_psr            5588
  firstk_nltk           5610
  centroid_mmr          5588
  centroid_mmr_bert     5588
  textrank             55894
  lexrank              55894
  bart                  5610
  pegasus               5610
  primera               5610
  qwen                  5610
  gemma                 5610
  mistral               5610
  gpt5mini              5471
  lsa                   5588
  lsa_steinberger       5588
  sbert_kmeans          5588
  sbert_agglom          5588
  lda                   5588



Lunghezza sorgente (parole): mediana 1288, max 35,362
Righe troncate a 3500 parole: 478 / 5610 (8.5%)


## Pilota

La corsa completa e' ~100.600 chiamate **a pagamento** e molte ore (di cui 72.681 gia'
pagate e in cache: un rilancio giudica solo i metodi nuovi). Il pilota valida prompt,
parsing, cache e — soprattutto — il **costo empirico per giudizio** prima di impegnarsi.

Il campione e' costruito con seed fisso **piu' le due righe PEGASUS patologiche note**
(`51178`, ciclo di ripetizione del beam search; `56099`, probabile disallineamento
sorgente/riassunto — vedi le avvertenze sul METEOR nel README). Servono da controllo di
**discriminazione**: se G-Eval non assegna a quelle fluency/coherence bassissime, la metrica
non sta misurando nulla di utile.

Criteri di accettazione prima di lanciare la corsa completa:

- >= 95% di giudizi riusciti e **nessuna (o quasi) risposta vuota** — se il budget di reasoning
  si esaurisce, alzare `MAX_COMPLETION_TOKENS` **prima** della corsa lunga, non durante;
- `cached_tokens > 0` sui giudizi successivi al primo di ogni riga (prova che la prompt cache
  funziona: se e' 0, fermarsi e rivedere l'ordine dei messaggi);
- distribuzione **non degenere** (deviazione standard > 0 su ogni dimensione: un giudice che
  risponde sempre 4 e' inutile);
- ordinamento plausibile dei metodi (LLM astrattivi sopra le baseline posizionali);
- costo/giudizio misurato, estrapolato ai giudizi ancora da fare: e' il numero **go/no-go**.

In [5]:
# --- Pilota ------------------------------------------------------------------
RIGHE_PATOLOGICHE = [51178, 56099]   # PEGASUS: ripetizione beam search / mismatch

if PILOTA:
    per_id = {rid: r for r in righe for rid in [r[0]]}
    scelte = [per_id[rid] for rid in RIGHE_PATOLOGICHE if rid in per_id]
    resto = [r for r in righe if r[0] not in RIGHE_PATOLOGICHE]
    random.Random(42).shuffle(resto)
    righe_pilota = scelte + resto[:max(PILOTA - len(scelte), 0)]

    cache = su.CacheGiudizi(CACHE_PATH)
    riepilogo = su.giudica_geval_concorrente(
        righe_pilota, giudica_uno, cache, n_thread=N_THREAD,
        etichetta='pilota ', ogni=max(OGNI // 20, 20), prezzi=PREZZI)
    cache.chiudi()

    rimanenti = totale_giudizi - riepilogo['fatti'] - riepilogo['saltati']
    costo_giudizio = riepilogo['costo_per_giudizio']
    print('\n=== ESTRAPOLAZIONE ALLA CORSA COMPLETA ===')
    print(f"  giudizi totali        : {totale_giudizi:,}")
    print(f"  costo/giudizio misurato: ${costo_giudizio:.5f}")
    print(f"  COSTO STIMATO TOTALE  : ${costo_giudizio * totale_giudizi:.2f}")
    print(f"  ore stimate           : "
          f"{totale_giudizi / max(riepilogo['giudizi_al_s'], 1e-9) / 3600:.1f} h "
          f"(a {N_THREAD} thread)")
    print(f"  token reasoning/giudizio: "
          f"{riepilogo['reasoning_tokens'] / max(riepilogo['giudizi'], 1):.0f}")
    print(f"  quota input in cache   : {riepilogo['quota_cached']:.1%}"
          f"   <- se e' ~0, l'ordine del prompt e' sbagliato")
else:
    print('Pilota non richiesto (GEVAL_PILOTA non impostata).')

Pilota non richiesto (GEVAL_PILOTA non impostata).


In [6]:
# --- Controllo di discriminazione sulle righe patologiche --------------------
if PILOTA:
    cache = su.CacheGiudizi(CACHE_PATH)
    for rid in RIGHE_PATOLOGICHE:
        voci = {m: cache.per_metodo(m).get(rid) for m in METODI}
        if not any(voci.values()):
            continue
        print(f'\nrow_id={rid}')
        for metodo, v in voci.items():
            if v:
                print(f"  {metodo:20s} coh={v['geval_coherence']} "
                      f"cons={v['geval_consistency']} flu={v['geval_fluency']} "
                      f"rel={v['geval_relevance']}")
    print('\nAtteso: PEGASUS con fluency/coherence molto basse (1-2) su queste righe.')

    # distribuzione dei punteggi: il giudice deve discriminare, non rispondere sempre 4
    import statistics
    tutti = [v for m in METODI for v in cache.per_metodo(m).values()]
    if tutti:
        print(f'\nDistribuzione su {len(tutti)} giudizi:')
        for col in su.COLONNE_METRICHE_GEVAL:
            valori = [v[col] for v in tutti]
            dev = statistics.pstdev(valori)
            stato = 'OK' if dev > 0 else 'DEGENERE'
            print(f'  {col:20s} media {statistics.mean(valori):.2f}  '
                  f'dev.std {dev:.2f}  [{stato}]')
    cache.chiudi()

## Corsa completa

Riprendibile: ogni giudizio viene scritto e flushato sulla cache JSONL appena arriva, e un
rilancio salta le coppie `(metodo, row_id)` gia' presenti. Interrompere con Ctrl-C non perde
nulla di pagato.

**Monitoraggio della spesa.** Non esiste un'API Azure che riporti il costo in tempo reale:
Cost Management ha 8-24 h di ritardo. La fonte di verita' e' l'oggetto `usage` di ogni
risposta, che qui viene sommato da `su.ContatoreCosti` e stampato ogni `OGNI` giudizi
(default 1500) con: ritmo, TPM osservato, ETA, quota di input in cache, quota di token di
reasoning, costo diviso per voce e **proiezione a fine corsa**. Poiche' i conteggi finiscono
anche nella cache, la stessa stima si puo' rileggere **da un secondo terminale a corsa in
corso**, senza toccare l'API:

```
python scripts/run_geval.py --costo
```

`BUDGET_MASSIMO` (`--budget`) e' un tetto duro: al superamento la corsa si ferma in modo
pulito e basta rilanciare con un tetto piu' alto.

**Attenzione**: cambiare deployment, rubriche o troncamento **dopo** aver popolato la cache
mescolerebbe corse diverse. In quel caso cancellare prima `geval_cache_{SCOPE}.jsonl`.

In [7]:
# --- Corsa completa ----------------------------------------------------------
if not PILOTA and not SOLO_METRICHE:
    cache = su.CacheGiudizi(CACHE_PATH)
    print(f'Giudizi gia\' in cache: {len(cache):,} / {totale_giudizi:,}')
    if RIPROVA_ERRORI:
        rimasti = cache.dimentica_errori()
        print(f'Errori rimossi dalla cache: ora {rimasti:,} voci (verranno ritentati)')

    # Il budget e' un tetto COMPLESSIVO: quanto e' gia' costato cio' che sta in
    # cache va sommato a questa sessione, altrimenti ogni rilancio ripartirebbe
    # da zero e un tetto di $38 diventerebbe $38 a corsa.
    speso_prima = sum(su.costo_da_token(cache.totali_token(), PREZZI).values())
    if BUDGET_MASSIMO:
        print(f"Budget complessivo ${BUDGET_MASSIMO:.2f} — di cui ${speso_prima:.2f} "
              f"gia' spesi in cache, disponibili ${BUDGET_MASSIMO - speso_prima:.2f}")

    riepilogo = su.giudica_geval_concorrente(
        righe[:LIMIT_RIGHE], giudica_uno, cache, n_thread=N_THREAD,
        etichetta='', ogni=OGNI, prezzi=PREZZI, budget_massimo=BUDGET_MASSIMO,
        speso_precedente=speso_prima)
    cache.chiudi()

    print('\n' + json.dumps({k: v for k, v in riepilogo.items()
                             if k not in ('costo',)}, indent=2, default=str))
else:
    print('Corsa completa saltata (modalita\' pilota o solo-metriche).')

Giudizi gia' in cache: 90,233 / 100,621
Budget complessivo $100.00 — di cui $83.00 gia' spesi in cache, disponibili $17.00


--- 1503 giudizi pagati (67334 da cache, 31784 rimasti) ---
  ritmo   : 1.81 giudizi/s, TPM osservato 285,635, ETA 4.9 h
  input   : 3,907,299 token (38% in cache)
  output  : 44,142 token (0% di reasoning)
  costo   : $1.87 (input $1.60 + cached $0.10 + output $0.17) = $0.00124/giudizio
  PROIEZIONE a fine corsa: $41.39
  errori  : content_filter=18, content_filter_propagato=72


--- 3004 giudizi pagati (71537 da cache, 26080 rimasti) ---
  ritmo   : 1.82 giudizi/s, TPM osservato 296,328, ETA 4.0 h
  input   : 8,050,686 token (39% in cache)
  output  : 88,058 token (0% di reasoning)
  costo   : $3.80 (input $3.25 + cached $0.21 + output $0.35) = $0.00127/giudizio
  PROIEZIONE a fine corsa: $36.81
  errori  : content_filter=42, content_filter_propagato=167


--- 4506 giudizi pagati (75745 da cache, 20370 rimasti) ---
  ritmo   : 1.81 giudizi/s, TPM osservato 295,892, ETA 3.1 h
  input   : 12,168,386 token (39% in cache)
  output  : 132,130 token (0% di reasoning)
  costo   : $5.70 (input $4.87 + cached $0.32 + output $0.52) = $0.00127/giudizio
  PROIEZIONE a fine corsa: $31.48
  errori  : content_filter=65, content_filter_propagato=257


--- 6006 giudizi pagati (79924 da cache, 14691 rimasti) ---
  ritmo   : 1.81 giudizi/s, TPM osservato 294,435, ETA 2.3 h
  input   : 16,129,563 token (39% in cache)
  output  : 176,156 token (0% di reasoning)
  costo   : $7.58 (input $6.46 + cached $0.42 + output $0.70) = $0.00126/giudizio
  PROIEZIONE a fine corsa: $26.10
  errori  : content_filter=86, content_filter_propagato=341


--- 7506 giudizi pagati (84025 da cache, 9090 rimasti) ---
  ritmo   : 1.79 giudizi/s, TPM osservato 290,474, ETA 1.4 h
  input   : 20,034,328 token (39% in cache)
  output  : 219,930 token (0% di reasoning)
  costo   : $9.40 (input $8.02 + cached $0.52 + output $0.87) = $0.00125/giudizio
  PROIEZIONE a fine corsa: $20.79
  errori  : content_filter=100, content_filter_propagato=397


--- 9007 giudizi pagati (88318 da cache, 3296 rimasti) ---
  ritmo   : 1.79 giudizi/s, TPM osservato 289,827, ETA 0.5 h
  input   : 24,093,869 token (39% in cache)
  output  : 263,990 token (0% di reasoning)
  costo   : $11.31 (input $9.64 + cached $0.62 + output $1.04) = $0.00126/giudizio
  PROIEZIONE a fine corsa: $15.44
  errori  : content_filter=130, content_filter_propagato=516


--- 9717 giudizi pagati (90233 da cache, 671 rimasti) ---
  ritmo   : 1.79 giudizi/s, TPM osservato 289,478, ETA 0.1 h
  input   : 25,976,241 token (39% in cache)
  output  : 284,667 token (0% di reasoning)
  costo   : $12.19 (input $10.39 + cached $0.67 + output $1.13) = $0.00125/giudizio
  PROIEZIONE a fine corsa: $13.03
  errori  : content_filter=135, content_filter_propagato=536

{
  "fatti": 9717,
  "saltati": 90233,
  "rimasti": 671,
  "totale_giudizi": 100621,
  "fermato_per_budget": false,
  "errori": {
    "content_filter": 135,
    "content_filter_propagato": 536
  },
  "giudizi": 9717,
  "durata_s": 5443.0904178619385,
  "giudizi_al_s": 1.7851990788381695,
  "tpm_osservato": 289477.91769715294,
  "prompt_tokens": 25976241,
  "cached_tokens": 10211584,
  "completion_tokens": 284667,
  "reasoning_tokens": 0,
  "quota_cached": 0.3931124599590834,
  "quota_reasoning": 0.0,
  "costo_totale": 12.1873687332,
  "costo_per_giudizio": 0.0012542316284038283
}


## Scrittura delle metriche

Passo **separato e a costo zero**: rilegge solo la cache, senza alcuna chiamata API — lo stesso
principio del resto del repository, dove la valutazione legge esclusivamente i file salvati.
Puo' quindi essere ripetuto quante volte serve (`scripts/run_geval.py --solo-metriche`), anche
a corsa non finita, per avere una fotografia parziale.

La `config` storica di ciascun metodo **non** viene toccata: questi sono file nuovi, separati
da `{metodo}_{scope}_aggregate.json`, e la `config` che ci scriviamo dentro e' quella del
**giudice**, non quella della corsa di generazione.

In [8]:
# --- Scrittura dei file di metrica -------------------------------------------
cache = su.CacheGiudizi(CACHE_PATH)
errori_per_metodo = Counter(m for m, _, _ in cache.errori())

riepilogo_metodi = {}
for metodo in METODI:
    punteggi = cache.per_metodo(metodo)
    if not punteggi:
        print(f'({metodo}: nessun giudizio in cache, salto)')
        continue
    n_attesi = sum(1 for _, _, coppie in righe if metodo in dict(coppie))
    diagnostica = {
        'n_giudizi_riusciti': len(punteggi),
        'n_richiesti': n_attesi,
        'n_errori': errori_per_metodo.get(metodo, 0),
        'copertura': round(len(punteggi) / n_attesi, 4) if n_attesi else 0.0,
    }
    _, aggregato = su.salva_metriche_geval(punteggi, metodo, SCOPE, P['metrics_dir'],
                                           config, diagnostica=diagnostica)
    riepilogo_metodi[metodo] = aggregato

print('\nG-Eval medio per metodo (scala 1-5):')
print(f"{'metodo':22s} {'n':>6s} {'coher':>6s} {'consi':>6s} {'fluen':>6s} "
      f"{'relev':>6s} {'MEDIA':>6s}")
for metodo, agg in sorted(riepilogo_metodi.items(),
                          key=lambda kv: -kv[1]['overall']['geval_media']):
    o = agg['overall']
    print(f"{metodo:22s} {agg['n_esempi']:6d} {o['geval_coherence']:6.2f} "
          f"{o['geval_consistency']:6.2f} {o['geval_fluency']:6.2f} "
          f"{o['geval_relevance']:6.2f} {o['geval_media']:6.2f}")

G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\firstk_psr_test_geval_per_example.csv (5229 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\firstk_psr_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\firstk_nltk_test_geval_per_example.csv (5250 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\firstk_nltk_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\centroid_mmr_test_geval_per_example.csv (5230 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\centroid_mmr_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-new

G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\lexrank_test_geval_per_example.csv (5229 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\lexrank_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\bart_test_geval_per_example.csv (5249 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\bart_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\pegasus_test_geval_per_example.csv (5247 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\pegasus_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\result

G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\gemma_test_geval_per_example.csv (5244 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\gemma_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\mistral_test_geval_per_example.csv (5244 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\mistral_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\gpt5mini_test_geval_per_example.csv (5215 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\gpt5mini_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\re

G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\sbert_kmeans_test_geval_per_example.csv (5325 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\sbert_kmeans_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\sbert_agglom_test_geval_per_example.csv (5305 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\sbert_agglom_test_geval_aggregate.json
G-Eval per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\lda_test_geval_per_example.csv (5294 righe)
G-Eval aggregato   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\lda_test_geval_aggregate.json

G-Eval medio per metodo (scala 1-5):
metodo                      n  coher  consi 

## Diagnostica e meta-valutazione

Tre controlli, nell'ordine in cui contano:

1. **il giudice discrimina?** Se la deviazione standard di una dimensione e' ~0, il modello sta
   rispondendo sempre lo stesso numero e quella dimensione va buttata.
2. **cosa e' andato storto?** Errori raggruppati per tipo. Le righe respinte dal content filter
   di Azure non sono ritentabili senza un filtro custom *high-only* attaccato al deployment.
   Dentro una singola corsa il rifiuto e' quasi interamente **determinato dalla sorgente** (nella
   corsa dei tredici metodi: unione 366 righe respinte, intersezione 358 — cioe' praticamente le
   stesse per ogni metodo, come ci si aspetta da un filtro che guarda il testo condiviso). **Fra
   corse diverse invece no**: nel backfill dei cinque metodi dei notebook 15-17, due settimane
   dopo, 155 di quelle righe passano, 91 nuove vengono respinte e la varianza fra metodi cresce
   (unione 294, intersezione 187). La copertura piu' alta dei cinque (94,7-96,7% contro
   93,5-93,6%) e' quindi una proprieta' del **filtro al momento della corsa**, non dei metodi.
3. **G-Eval dice qualcosa di nuovo?** La correlazione di Spearman fra `geval_media` e
   `rouge1_f1`/`bertscore_f1` e' il vero contributo: se fosse ~1, la metrica sarebbe ridondante;
   dove diverge, sta misurando qualcosa che le metriche lessicali non vedono.

In [9]:
# --- Diagnostica -------------------------------------------------------------
import statistics

import pandas as pd

tutti = [v for m in METODI for v in cache.per_metodo(m).values()]
print(f'Distribuzione dei punteggi su {len(tutti):,} giudizi:')
for col in su.COLONNE_METRICHE_GEVAL:
    valori = [v[col] for v in tutti]
    dev = statistics.pstdev(valori)
    print(f'  {col:20s} media {statistics.mean(valori):.2f}  dev.std {dev:.2f}  '
          f'[{"OK" if dev > 0.05 else "SOSPETTO: giudice poco discriminante"}]')

errori = cache.errori()
print(f'\nGiudizi falliti in cache: {len(errori):,}')
if errori:
    tipi = Counter('content_filter' if 'content_filter' in e.lower() else e[:60]
                   for _, _, e in errori)
    for tipo, n in tipi.most_common(10):
        print(f'  {n:6d}  {tipo}')

# Correlazione con le metriche esistenti
print('\nCorrelazione di Spearman (per metodo, sulle righe in comune):')
print(f"{'metodo':22s} {'n':>6s} {'vs ROUGE-1':>11s} {'vs BERTScore':>13s}")
for metodo in METODI:
    geval_path = P['metrics_dir'] / f'{metodo}_{SCOPE}_geval_per_example.csv'
    std_path = P['metrics_dir'] / f'{metodo}_{SCOPE}_per_example.csv'
    if not (geval_path.exists() and std_path.exists()):
        continue
    df = pd.read_csv(std_path).merge(pd.read_csv(geval_path), on='row_id', how='inner')
    if len(df) < 50:
        continue
    r_rouge = df['geval_media'].corr(df['rouge1_f1'], method='spearman')
    r_bert = (df['geval_media'].corr(df['bertscore_f1'], method='spearman')
              if 'bertscore_f1' in df.columns else float('nan'))
    print(f'{metodo:22s} {len(df):6d} {r_rouge:11.3f} {r_bert:13.3f}')

# Ispezione qualitativa: i giudizi piu' bassi
print('\n10 giudizi piu\' bassi (ispezione qualitativa):')
peggiori = sorted(((v['geval_media'], m, rid)
                   for m in METODI for rid, v in cache.per_metodo(m).items()))[:10]
for media, metodo, rid in peggiori:
    testo = riassunti[metodo].get(rid, '')
    print(f"\n  [{media:.2f}] {metodo} row_id={rid}")
    print(f"  {testo[:300]}")
cache.chiudi()

Distribuzione dei punteggi su 94,766 giudizi:
  geval_coherence      media 3.34  dev.std 1.35  [OK]
  geval_consistency    media 3.18  dev.std 1.40  [OK]
  geval_fluency        media 3.22  dev.std 1.41  [OK]
  geval_relevance      media 3.73  dev.std 1.05  [OK]


  geval_media          media 3.37  dev.std 1.16  [OK]



Giudizi falliti in cache: 5,855
    5855  content_filter

Correlazione di Spearman (per metodo, sulle righe in comune):
metodo                      n  vs ROUGE-1  vs BERTScore


firstk_psr               5229       0.149         0.378
firstk_nltk              5250       0.139         0.453
centroid_mmr             5230       0.147         0.307
centroid_mmr_bert        5230       0.147         0.309
textrank                 5230       0.080         0.260
lexrank                  5229       0.108         0.272
bart                     5249       0.274         0.337
pegasus                  5247       0.338         0.434
primera                  5247       0.226         0.324
qwen                     5244       0.136         0.311
gemma                    5244       0.238         0.317
mistral                  5244       0.142         0.295


gpt5mini                 5215       0.170         0.245
lsa                      5401       0.331         0.477
lsa_steinberger          5353       0.212         0.368
sbert_kmeans             5325       0.229         0.371
sbert_agglom             5305       0.357         0.491
lda                      5294       0.210         0.359

10 giudizi piu' bassi (ispezione qualitativa):



  [1.00] bart row_id=52149
  During Charles Darwin’s journey to the Galápion, he noted the existence of “a curious group of ‘a a curious group of  a hundred’ and that they are at least some of the best examples of “a magnitude’” of the world. “The babies can’t be found on every island in the world,” he said, “This is like a rea

  [1.00] bart row_id=54965
  ’s of v and v can be v, v and and a v is a and the v of and an v-and that v” is and ’v and a virtual is not a can’t be the an and not the state of the can't be a ‘v and the nouva of a pland, the i's is a van and a nation of the former of a “v and an ” and a ‘v-v is the and former of the  v that is a

  [1.00] centroid_mmr row_id=50548
  "I've done my time," Simpson said Hide Caption 5 of 23 Photos: The rise and fall of O.J. Simpson Simpson married Nicole Brown Simpson in 1985 Hide Caption 6 of 23 Photos: The rise and fall of O.J. Simpson Coach Lou Sabin and O.J. Simpson Hide Caption 7 of 23 Photos: The rise and fall of O.J. Simpson

## Validazione consigliata prima della corsa completa

Scala a tre gradini, dal piu' economico (`scripts/run_geval.py` imposta le variabili
d'ambiente per conto suo):

1. **smoke a 1 riga** — `python scripts/run_geval.py --righe 1` (una chiamata per ogni
   metodo non ancora in cache su quella riga). Verifica
   endpoint, deployment, chiave, conformita' del JSON, assenza di risposte vuote e soprattutto
   che `cached_tokens > 0` sulle chiamate successive alla prima: e' la prova empirica che la
   prompt cache funziona. Se e' 0, **fermarsi** e rivedere l'ordine dei messaggi prima di
   spendere altro.
2. **pilota a 20 righe** — `python scripts/run_geval.py --pilota 20`. Restituisce
   il costo/giudizio misurato ed estrapolato ai giudizi previsti, i token di reasoning effettivi, il TPM
   osservato e i criteri di accettazione elencati sopra.
3. **corsa completa** — `python scripts/run_geval.py --budget <tetto>`, riprendibile.

Dopo la corsa completa, ri-eseguire il **notebook 05** per aggiornare le viste di confronto con
le nuove colonne G-Eval, e verificare con `git status` che i CSV per-esempio **standard**
risultino immutati: e' il controllo diretto sul fatto che il backfill non abbia toccato le
metriche gia' pubblicate.